# 02_Token — Análise de sentimentos com BoW e TF-IDF

**Aluno: Tiago Galhardo Avelar**

Esta é uma cópia resolvida do notebook-modelo da aula. O objetivo é treinar um classificador binário de sentimentos no IMDb e comparar, sob a mesma divisão de dados e o mesmo classificador, três representações de entrada:

1. Bag-of-Words booleano;
2. Bag-of-Words por contagem (histograma);
3. TF-IDF.

A tokenização, o vocabulário e as três conversões para features são implementados manualmente com Python e NumPy. O scikit-learn é usado somente para treinar o classificador linear e medir a acurácia.


## 1. Configuração e reprodutibilidade

Seguindo a escala do `imdb_sample` usado no notebook-modelo, é selecionada de forma determinística uma amostra de 1.000 avaliações do conjunto oficial de treino do IMDb: 800 exemplos para treino e 200 para validação. O vocabulário contém os 3.000 tokens mais frequentes observados **somente no treino**, evitando vazamento de informação.


In [1]:
import gc
import html
import math
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import sklearn
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC

SEED = 123
VOCAB_SIZE = 3_000
TRAIN_SIZE = 800
VALID_SIZE = 200

random.seed(SEED)
np.random.seed(SEED)

print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Semente:", SEED)
print("Vocabulário máximo:", VOCAB_SIZE)


NumPy: 2.5.4
scikit-learn: 1.9.1
Semente: 123
Vocabulário máximo: 3000


## 2. Carregamento e divisão do IMDb

Os caminhos são ordenados antes da leitura e o embaralhamento usa uma instância local de `random.Random`, tornando a divisão reproduzível. As avaliações do diretório oficial de teste não são consultadas neste experimento.


In [2]:
DATA_DIR = Path("aclImdb")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Diretório aclImdb não encontrado. Extraia o dataset IMDb antes de executar."
    )


def load_labeled_reviews(root):
    pairs = []
    for label_name, label in (("neg", 0), ("pos", 1)):
        for path in sorted((root / label_name).glob("*.txt")):
            pairs.append((path.read_text(encoding="utf-8"), label))
    return pairs


examples = load_labeled_reviews(DATA_DIR / "train")
random.Random(SEED).shuffle(examples)

train_examples = examples[:TRAIN_SIZE]
valid_examples = examples[TRAIN_SIZE:TRAIN_SIZE + VALID_SIZE]

x_train = [text for text, _ in train_examples]
y_train = np.asarray([label for _, label in train_examples], dtype=np.int8)
x_valid = [text for text, _ in valid_examples]
y_valid = np.asarray([label for _, label in valid_examples], dtype=np.int8)

assert len(x_train) == TRAIN_SIZE
assert len(x_valid) == VALID_SIZE
assert set(np.unique(y_train)) == {0, 1}

print("Treino:", len(x_train), dict(zip(*np.unique(y_train, return_counts=True))))
print("Validação:", len(x_valid), dict(zip(*np.unique(y_valid, return_counts=True))))
print("Exemplo:", x_train[0][:180].replace("\n", " "))


Treino: 800 {np.int8(0): np.int64(412), np.int8(1): np.int64(388)}
Validação: 200 {np.int8(0): np.int64(93), np.int8(1): np.int64(107)}
Exemplo: Remember Ralph Bakshi? The guy that was an animator on Terrytoons, then on Paramount Cartoon Studios, after that, he was a director on Fritz the Cat 1 & 2 and Heavy Traffic? Well, 


## 3. Tokenizador implementado manualmente

A função abaixo usa apenas módulos nativos: converte entidades HTML, remove tags, transforma em minúsculas e encontra palavras que podem conter apóstrofo. Números e pontuação isolada são descartados. Nenhum tokenizador pronto é utilizado.


In [3]:
TAG_RE = re.compile(r"<[^>]+>")
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")


def tokenize(text):
    clean = html.unescape(text)
    clean = TAG_RE.sub(" ", clean)
    return TOKEN_RE.findall(clean.lower())


token_example = tokenize("<br />This movie isn't bad; it is surprisingly GOOD! 10/10")
print(token_example)
assert token_example == ["this", "movie", "isn't", "bad", "it", "is", "surprisingly", "good"]


['this', 'movie', "isn't", 'bad', 'it', 'is', 'surprisingly', 'good']


## 4. Vocabulário de treino

A contagem global é feita com `collections.Counter`. Empates são resolvidos alfabeticamente antes da seleção dos termos, garantindo estabilidade entre execuções.


In [4]:
start = time.perf_counter()
token_counts = Counter()
for text in x_train:
    token_counts.update(tokenize(text))

ranked_tokens = sorted(token_counts.items(), key=lambda item: (-item[1], item[0]))
vocabulary = {
    token: index
    for index, (token, _) in enumerate(ranked_tokens[:VOCAB_SIZE])
}

assert len(vocabulary) == VOCAB_SIZE
assert len(set(vocabulary.values())) == VOCAB_SIZE
print("Tokens distintos no treino:", len(token_counts))
print("Primeiros termos:", ranked_tokens[:15])
print(f"Vocabulário criado em {time.perf_counter() - start:.2f} s")


Tokens distintos no treino: 16171
Primeiros termos: [('the', 10663), ('and', 5112), ('a', 5094), ('of', 4454), ('to', 4279), ('is', 3477), ('in', 2980), ('it', 2583), ('i', 2573), ('this', 2448), ('that', 2288), ('was', 1600), ('as', 1525), ('but', 1413), ('movie', 1356)]
Vocabulário criado em 0.05 s


## 5. Conversão manual para histograma

Uma única matriz de contagens inteiras é construída para as 1.000 avaliações. Para cada documento, os tokens são procurados no dicionário do vocabulário e incrementados manualmente. As outras duas representações serão derivadas dessa matriz por operações NumPy.


In [5]:
def texts_to_count_matrix(texts, token_to_index):
    matrix = np.zeros((len(texts), len(token_to_index)), dtype=np.uint16)
    for row, text in enumerate(texts):
        document_counts = Counter(tokenize(text))
        for token, count in document_counts.items():
            column = token_to_index.get(token)
            if column is not None:
                matrix[row, column] = min(count, np.iinfo(np.uint16).max)
    return matrix


start = time.perf_counter()
all_texts = x_train + x_valid
count_matrix = texts_to_count_matrix(all_texts, vocabulary)

assert count_matrix.shape == (TRAIN_SIZE + VALID_SIZE, VOCAB_SIZE)
assert count_matrix.dtype == np.uint16
print("Forma da matriz de contagens:", count_matrix.shape)
print("Memória da matriz: %.1f MB" % (count_matrix.nbytes / 1024**2))
print(f"Conversão concluída em {time.perf_counter() - start:.2f} s")


Forma da matriz de contagens: (1000, 3000)
Memória da matriz: 5.7 MB
Conversão concluída em 0.18 s


## 6. Implementações das três features

- **BoW booleano:** $x_{d,t}=1$ se o token $t$ aparece no documento $d$; caso contrário, zero.
- **Histograma:** $x_{d,t}$ é o número de ocorrências de $t$ em $d$.
- **TF-IDF:** $tf(t,d)=count(t,d)/\sum_j count(j,d)$ e $idf(t)=\log((N+1)/(df(t)+1))+1$. O vetor final é normalizado pela norma L2.

O IDF é calculado exclusivamente sobre as 800 amostras de treino e depois aplicado à validação.


In [6]:
def boolean_bow(counts):
    return (counts > 0).astype(np.float32)


def histogram_bow(counts):
    return counts.astype(np.float32)


def fit_idf(train_counts):
    document_frequency = np.count_nonzero(train_counts, axis=0)
    return (np.log((train_counts.shape[0] + 1) / (document_frequency + 1)) + 1).astype(np.float32)


def tf_idf(counts, idf):
    values = counts.astype(np.float32)
    document_lengths = values.sum(axis=1, keepdims=True)
    np.divide(values, document_lengths, out=values, where=document_lengths != 0)
    values *= idf
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    np.divide(values, norms, out=values, where=norms != 0)
    return values


toy = np.asarray([[2, 0, 1], [0, 1, 1]], dtype=np.uint16)
toy_idf = fit_idf(toy)
print("BoW booleano:\n", boolean_bow(toy))
print("Histograma:\n", histogram_bow(toy))
print("TF-IDF:\n", np.round(tf_idf(toy, toy_idf), 4))

assert np.array_equal(boolean_bow(toy), [[1, 0, 1], [0, 1, 1]])
assert np.array_equal(histogram_bow(toy), toy.astype(np.float32))
assert np.allclose(np.linalg.norm(tf_idf(toy, toy_idf), axis=1), 1.0)
print("Testes unitários das features: OK")


BoW booleano:
 [[1. 0. 1.]
 [0. 1. 1.]]
Histograma:
 [[2. 0. 1.]
 [0. 1. 1.]]
TF-IDF:
 [[0.9422 0.     0.3352]
 [0.     0.8148 0.5797]]
Testes unitários das features: OK


## 7. Classificador e protocolo de avaliação

Para tornar a comparação justa, as três representações usam o mesmo SVM linear (`LinearSVC`), com a mesma configuração. O scikit-learn não participa da tokenização nem da produção das features.


In [7]:
def train_and_evaluate(name, features):
    classifier = LinearSVC(C=1.0, dual="auto", max_iter=10_000, random_state=SEED)
    start = time.perf_counter()
    classifier.fit(features[:TRAIN_SIZE], y_train)
    predictions = classifier.predict(features[TRAIN_SIZE:])
    elapsed = time.perf_counter() - start
    accuracy = accuracy_score(y_valid, predictions)
    result = {
        "feature": name,
        "accuracy": float(accuracy),
        "correct": int((predictions == y_valid).sum()),
        "total": int(len(y_valid)),
        "seconds": float(elapsed),
    }
    print(result)
    return result


results = []


### I — Bag-of-Words booleano


In [8]:
features = boolean_bow(count_matrix)
results.append(train_and_evaluate("BoW booleano", features))
del features
gc.collect()


{'feature': 'BoW booleano', 'accuracy': 0.815, 'correct': 163, 'total': 200, 'seconds': 0.05516869999701157}


0

### II — Bag-of-Words com contagem


In [9]:
features = histogram_bow(count_matrix)
results.append(train_and_evaluate("BoW contagem", features))
del features
gc.collect()


{'feature': 'BoW contagem', 'accuracy': 0.8, 'correct': 160, 'total': 200, 'seconds': 0.058779000013601035}


0

### III — TF-IDF


In [10]:
idf = fit_idf(count_matrix[:TRAIN_SIZE])
features = tf_idf(count_matrix, idf)
results.append(train_and_evaluate("TF-IDF", features))
del features
gc.collect()


{'feature': 'TF-IDF', 'accuracy': 0.83, 'correct': 166, 'total': 200, 'seconds': 0.025177799980156124}


0

## 8. Comparação dos resultados


In [11]:
print(f"{'Feature':<18} {'Corretos':>10} {'Total':>8} {'Acurácia':>12} {'Tempo (s)':>12}")
print("-" * 64)
for result in results:
    print(
        f"{result['feature']:<18} {result['correct']:>10} {result['total']:>8} "
        f"{100 * result['accuracy']:>11.2f}% {result['seconds']:>12.2f}"
    )

best_result = max(results, key=lambda item: item["accuracy"])
print("\nMelhor representação:", best_result["feature"])
print("Melhor acurácia: %.2f%%" % (100 * best_result["accuracy"]))

assert len(results) == 3
assert all(0.50 < item["accuracy"] <= 1.0 for item in results)


Feature              Corretos    Total     Acurácia    Tempo (s)
----------------------------------------------------------------
BoW booleano              163      200       81.50%         0.06
BoW contagem              160      200       80.00%         0.06
TF-IDF                    166      200       83.00%         0.03

Melhor representação: TF-IDF
Melhor acurácia: 83.00%


## 9. Discussão

As três representações preservam informações diferentes. O BoW booleano registra apenas presença/ausência e reduz o efeito de repetições. O histograma mantém frequência absoluta, mas textos longos podem produzir valores maiores. O TF-IDF combina frequência relativa com menor peso para termos presentes em muitos documentos, realçando palavras mais discriminativas.

A comparação é controlada: divisão de dados, vocabulário, classificador e hiperparâmetros são idênticos; somente a representação de entrada muda. As acurácias efetivamente observadas estão registradas na célula anterior.


## Ferramentas e referências

- Notebook-modelo da disciplina: [Aula 2 — Bag of Words e TF-IDF](https://colab.research.google.com/drive/1h5dRZ1eWTRDaG6kh9C044hhEGNTqvYB2?usp=sharing).
- Maas et al. (2011), dataset: [Learning Word Vectors for Sentiment Analysis](https://ai.stanford.edu/~amaas/data/sentiment/).
- NumPy: [documentação oficial](https://numpy.org/doc/stable/).
- Python: [`re`](https://docs.python.org/3/library/re.html), [`html`](https://docs.python.org/3/library/html.html) e [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter).
- scikit-learn: [`LinearSVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html) e [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html).
- Jupyter `nbclient`: [execução programática de notebooks](https://nbclient.readthedocs.io/).


## Comentários

---
**Comentários:**

Ferramenta: OpenAI Codex (modelo GPT-5)

Utilização: Utilizado como apoio para interpretar o notebook-modelo, implementar e revisar as três representações sem vetorizadores prontos, documentar as referências e validar o notebook executado. As métricas apresentadas são produzidas pela execução efetiva das células.

Ferramenta: Python e NumPy

Utilização: Utilizados para tokenização, construção do vocabulário e implementação manual das features BoW booleano, histograma e TF-IDF.

Ferramenta: scikit-learn

Utilização: Utilizado exclusivamente para treinar o classificador linear e calcular a acurácia, conforme permitido no enunciado.

Ferramenta: Jupyter Notebook / nbclient

Utilização: Utilizado para executar todas as células e registrar as evidências no arquivo entregue.
---


## Conclusão

Foi treinado e avaliado um classificador binário de sentimentos com três tipos de features implementadas manualmente. O experimento mantém dados, vocabulário e classificador constantes para isolar o impacto da representação. A tabela executada registra as acurácias e identifica automaticamente a melhor alternativa neste conjunto de validação.
